In [ ]:
!mkdir -p /kaggle/tmp
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI /kaggle/tmp/ComfyUI
%cd /kaggle/tmp/ComfyUI
!pip install -q -r requirements.txt

# Reference Latent+ (third-party custom node).
#
# Deliberately reintroducing a third-party node -- the dependency class this
# project dropped when it moved from UNO to core-node-only USO -- because
# core USO conditioning hit a measured ceiling on face consistency: identity
# plateaus at denoise 0.40 (0.500 -> 0.722) and raising denoise past that
# gained nothing while composition preservation fell 0.844 -> 0.799. The same
# character still measured a 0.326 identity spread across framings and read as
# a visibly different actor between shots.
#
# This node adds the axis core nodes do not expose: per-reference STRENGTH
# (default 0.85, up to 50) and timestep gating (start_percent/end_percent), so
# identity can be pushed hard while denoise -- and therefore composition --
# stays low. Its optional auto-masking needs mediapipe; explicit mask inputs
# override it, so masking is not depended on (MediaPipe cannot detect profile
# faces, which is what defeated the earlier crop-based identity pass).
!git clone --depth 1 https://github.com/shootthesound/comfyui-ReferenceLatentPlus /kaggle/tmp/ComfyUI/custom_nodes/comfyui-ReferenceLatentPlus
!pip install -q mediapipe
!ls /kaggle/tmp/ComfyUI/custom_nodes/
!df -h /kaggle/tmp


In [ ]:
import os
print('--- /kaggle/input contents (diagnostic) ---')
for root, dirs, files in os.walk('/kaggle/input'):
    for name in files:
        print(os.path.join(root, name))

token_path = '/kaggle/input/hf-token/hf_token.txt'
if not os.path.exists(token_path):
    print(f'Expected path {token_path} missing -- searching /kaggle/input for any hf_token*.txt file...')
    found = None
    for root, dirs, files in os.walk('/kaggle/input'):
        for name in files:
            if 'hf_token' in name.lower() or 'hf-token' in name.lower():
                found = os.path.join(root, name)
                break
        if found:
            break
    if not found:
        raise RuntimeError(
            'No hf_token file found anywhere under /kaggle/input -- the hf-token dataset is '
            'genuinely not mounted into this session at all (see the directory listing printed '
            'above for what IS mounted). This is not a path-naming mismatch.'
        )
    print(f'Found it at a different path than expected: {found}')
    token_path = found

with open(token_path) as f:
    os.environ['HF_TOKEN'] = f.read().strip()
from huggingface_hub import hf_hub_download

os.makedirs('models/unet', exist_ok=True)
os.makedirs('models/clip', exist_ok=True)
os.makedirs('models/vae', exist_ok=True)
os.makedirs('models/checkpoints', exist_ok=True)
os.makedirs('models/loras', exist_ok=True)

print('Downloading Flux.1-dev unet checkpoint (gated, ~23GB) into /kaggle/tmp (60GB budget, not the 20GB /kaggle/working cap)...')
hf_hub_download(repo_id='black-forest-labs/FLUX.1-dev', filename='flux1-dev.safetensors',
                 local_dir='models/unet', token=os.environ['HF_TOKEN'])

print('Downloading VAE...')
hf_hub_download(repo_id='black-forest-labs/FLUX.1-dev', filename='ae.safetensors',
                 local_dir='models/vae', token=os.environ['HF_TOKEN'])

print('Downloading text encoders (ungated)...')
hf_hub_download(repo_id='comfyanonymous/flux_text_encoders', filename='clip_l.safetensors', local_dir='models/clip')
hf_hub_download(repo_id='comfyanonymous/flux_text_encoders', filename='t5xxl_fp8_e4m3fn.safetensors', local_dir='models/clip')

print('Downloading USO all-in-one fp8 checkpoint (~16GB, public/ungated)...')
hf_hub_download(repo_id='Comfy-Org/flux1-dev', filename='flux1-dev-fp8.safetensors', local_dir='models/checkpoints')

print('Downloading USO LoRA (public/ungated)...')
hf_hub_download(repo_id='Comfy-Org/USO_1.0_Repackaged', filename='split_files/loras/uso-flux1-dit-lora-v1.safetensors',
                 local_dir='models/loras')
# hf_hub_download preserves the repo's internal subfolder structure under
# local_dir by default -- ComfyUI's LoraLoaderModelOnly needs the file
# directly in models/loras, not nested under split_files/loras/.
uso_lora_nested = 'models/loras/split_files/loras/uso-flux1-dit-lora-v1.safetensors'
uso_lora_flat = 'models/loras/uso-flux1-dit-lora-v1.safetensors'
if os.path.exists(uso_lora_nested) and not os.path.exists(uso_lora_flat):
    os.rename(uso_lora_nested, uso_lora_flat)
    print(f'Moved {uso_lora_nested} -> {uso_lora_flat}')

print('--- models/ tree ---')
!find models -type f -exec ls -lh {} \;
!df -h /kaggle/tmp


In [ ]:
import subprocess, time
print('--- baseline system RAM before first launch ---')
!free -h
log_file = open('/kaggle/tmp/comfyui.log', 'w')
proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188'],
    cwd='/kaggle/tmp/ComfyUI', stdout=log_file, stderr=subprocess.STDOUT,
)
print(f'ComfyUI starting, pid={proc.pid} - waiting 45s for startup...')
time.sleep(45)
print('--- last 60 lines of comfyui.log ---')
!tail -60 /kaggle/tmp/comfyui.log

watchdog_script = '''#!/bin/bash
LOG=/kaggle/tmp/watchdog.log
exec > >(tee -a "$LOG") 2>&1
cd /kaggle/tmp/ComfyUI
echo "=== WATCHDOG started $(date -u +%FT%TZ) ==="
while true; do
  sleep 15
  if ! curl -s -o /dev/null -m 5 http://localhost:8188/object_info; then
    if ! pgrep -f "python main.py --listen" > /dev/null; then
      echo "=== WATCHDOG $(date -u +%FT%TZ) ComfyUI process is gone, diagnosing before restart ==="
      echo "--- free -h ---"
      free -h
      echo "--- nvidia-smi memory ---"
      nvidia-smi --query-gpu=memory.used,memory.total --format=csv
      echo "--- tail of comfyui.log at time of death ---"
      tail -40 /kaggle/tmp/comfyui.log
      echo "=== WATCHDOG restarting ComfyUI ==="
      nohup python main.py --listen 0.0.0.0 --port 8188 >> /kaggle/tmp/comfyui.log 2>&1 &
      sleep 45
      echo "=== WATCHDOG restart attempt complete, resuming health polling ==="
    fi
  fi
done
'''
with open('/kaggle/tmp/watchdog.sh', 'w') as f:
    f.write(watchdog_script)
import os
os.chmod('/kaggle/tmp/watchdog.sh', 0o755)
subprocess.Popen(['bash', '/kaggle/tmp/watchdog.sh'], start_new_session=True)
print('Watchdog launched (detached) - will auto-restart ComfyUI on crash and log free-h/nvidia-smi to /kaggle/tmp/watchdog.log at the moment of death.')


In [ ]:
import requests, json, time
for attempt in range(6):
    try:
        r = requests.get('http://localhost:8188/object_info', timeout=10)
        if r.status_code == 200:
            break
    except Exception as e:
        print(f'attempt {attempt+1}: {e}')
    time.sleep(10)
else:
    raise RuntimeError('ComfyUI never became healthy - check comfyui.log above.')

info = r.json()
needed = ['CheckpointLoaderSimple', 'LoraLoaderModelOnly', 'ConditioningZeroOut',
          'ImageScaleToMaxDimension', 'VAEEncode', 'ReferenceLatent',
          'FluxKontextMultiReferenceLatentMethod', 'ImageScale']
missing = [n for n in needed if n not in info]
if missing:
    raise RuntimeError(f'This ComfyUI build is missing core node types: {missing}')
print(f'All {len(needed)} core node types present.')

# Find Reference Latent+ by display name, since its registered class_type key
# is not documented -- guessing node/input names is exactly how earlier graph
# integrations in this project broke, so the real schema is dumped here and
# read back rather than assumed.
matches = {
    key: meta for key, meta in info.items()
    if 'referencelatent' in key.lower().replace('_', '').replace(' ', '')
    or 'reference latent+' in str(meta.get('display_name', '')).lower()
}
print('\n=== Reference-latent-ish nodes registered ===')
for key, meta in matches.items():
    print(f'  class_type: {key!r}  display_name: {meta.get("display_name")!r}')

plus = {k: v for k, v in matches.items() if k != 'ReferenceLatent'}
if not plus:
    raise RuntimeError(
        'Reference Latent+ did not register. Check the custom_nodes clone and the '
        'ComfyUI startup log for an import error -- proceeding without it would '
        'silently fall back to the weaker core-only identity path.'
    )
for key, meta in plus.items():
    print(f'\n=== FULL SCHEMA for {key} ===')
    print(json.dumps(meta, indent=2)[:4000])


In [ ]:
import subprocess, re, time, os, json, datetime

!curl -Ls https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /kaggle/tmp/cloudflared
!chmod +x /kaggle/tmp/cloudflared

# cloudflared runs in the BACKGROUND rather than blocking this cell, so the
# URL it prints can be captured and published before we settle into the
# keep-alive wait.
#
# Why publishing is necessary: a Kaggle batch kernel only commits its logs
# and output when the run ENDS -- verified directly, `kaggle kernels logs`
# and `kaggle kernels output` both return empty while a kernel is RUNNING.
# But this cell must never end (the tunnel, and therefore ComfyUI's only
# route to the outside world, dies with it). That makes the tunnel URL
# permanently unreachable from the Kaggle CLI. Writing it into a tiny
# private dataset is what lets the local backend discover it automatically
# instead of a human copy-pasting it out of the notebook UI every restart.
TUNNEL_LOG = '/kaggle/tmp/cloudflared.log'
log_handle = open(TUNNEL_LOG, 'w')
proc = subprocess.Popen(
    ['/kaggle/tmp/cloudflared', 'tunnel', '--url', 'http://localhost:8188'],
    stdout=log_handle, stderr=subprocess.STDOUT,
)

url = None
for _ in range(60):
    time.sleep(5)
    try:
        with open(TUNNEL_LOG) as f:
            content = f.read()
    except FileNotFoundError:
        continue
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', content)
    if match:
        url = match.group(0)
        break

if not url:
    print('--- cloudflared log so far ---')
    !tail -40 /kaggle/tmp/cloudflared.log
    raise RuntimeError('cloudflared never printed a trycloudflare.com URL.')

print('=== TUNNEL URL: ' + url + ' ===')

# --- publish the URL to a tiny private dataset -------------------------
cred_path = '/kaggle/input/kaggle-creds/access_token'
if not os.path.exists(cred_path):
    found = None
    for root, dirs, files in os.walk('/kaggle/input'):
        for name in files:
            if 'access_token' in name.lower():
                found = os.path.join(root, name)
                break
        if found:
            break
    cred_path = found

if cred_path:
    os.environ['KAGGLE_API_TOKEN'] = cred_path
    pub_dir = '/kaggle/tmp/tunnel_pub'
    os.makedirs(pub_dir, exist_ok=True)
    stamp = datetime.datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
    with open(os.path.join(pub_dir, 'tunnel_url.txt'), 'w') as f:
        f.write(url + '\n' + stamp + '\n')
    with open(os.path.join(pub_dir, 'dataset-metadata.json'), 'w') as f:
        json.dump({
            'title': 'ComfyUI Tunnel URL',
            'id': 'krkomaleswarreddy/comfyui-tunnel-url',
            'licenses': [{'name': 'CC0-1.0'}],
        }, f)
    # VERSION first: the dataset already exists after the first ever run, and
    # `create` then fails ("title already in use"). The old create-then-version
    # order hid the URL on 2026-09-30 -- the version fallback failed silently
    # because its return code was never checked.
    r = subprocess.run(
        ['kaggle', 'datasets', 'version', '-p', pub_dir, '-m', 'tunnel url ' + stamp],
        capture_output=True, text=True,
    )
    print('datasets version rc=%d' % r.returncode, r.stdout, r.stderr)
    if r.returncode != 0:
        r = subprocess.run(['kaggle', 'datasets', 'create', '-p', pub_dir], capture_output=True, text=True)
        print('datasets create rc=%d' % r.returncode, r.stdout, r.stderr)
    if r.returncode != 0:
        print('!!!!! TUNNEL URL NOT PUBLISHED -- copy it from the line above: ' + url + ' !!!!!')
else:
    print('WARNING: kaggle-creds not mounted -- URL not published, read it from this log manually.')

# Keep the session (and therefore ComfyUI and the tunnel) alive.
print('Tunnel live, ComfyUI reachable. Holding session open...')
proc.wait()
